In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [6]:
BIELIK = "/home/test/caise-ner/ncgft/models/Bielik-11B-v3.0-Instruct" 
HERBERT = "/home/test/caise-ner/ncgft/models/herbert-base-cased"
LLAMA = "/home/test/caise-ner/ncgft/models/Llama3-OpenBioLLM-8B"
MEDGAMMA = "/home/test/caise-ner/ncgft/models/medgemma-27b-text-it"
MEDIPHI = "/home/test/caise-ner/ncgft/models/MediPhi-Instruct"
QWEN = "/home/test/caise-ner/ncgft/models/Qwen2.5-7B-Instruct"

In [7]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# po restarcie kernela najlepiej
gc.collect()
torch.cuda.empty_cache()

def load_local_4bit_gpu_only(local_path: str, trust_remote_code: bool = False):
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        local_path,
        local_files_only=True,
        trust_remote_code=trust_remote_code,
        use_fast=True,
    )

    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        local_path,
        local_files_only=True,
        trust_remote_code=trust_remote_code,
        quantization_config=quant_config,
        device_map={"": 0},   # wszystko na GPU
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )

    model.eval()
    return tokenizer, model

In [8]:
MODEL_PATH = QWEN

tokenizer, model = load_local_4bit_gpu_only(
    MODEL_PATH,
    trust_remote_code=False
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/home/test/anaconda3/envs/caise-ner/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [9]:
KEYWORDS = ["pulmonary nodule", "pleural effusion", "mediastinal lymph nodes"]

prompt = f"""
You are generating synthetic structured medical reports.

Task:
Generate exactly 5 examples.

Output rules:
- Return ONLY a valid JSON array.
- Do not include markdown.
- Do not include explanations.
- Do not include comments.
- Do not repeat the prompt.

Each array element must be an object with exactly these fields:
{{
  "id": integer,
  "ct_thorax": string,
  "findings": string,
  "technique": string,
  "consultation_report": string,
  "keywords_used": [string]
}}

Requirements for EACH example:
1. "ct_thorax", "findings", "technique", and "consultation_report" must be natural, coherent, and written in English.
2. The combined content of the example must include ALL of the following mandatory keywords exactly as written:
   {KEYWORDS}
3. "keywords_used" must contain all mandatory keywords.
4. Each example must be meaningfully different from the others.
5. Do not repeat the same sentence pattern.
6. Keep each field concise:
   - "ct_thorax": 30 to 60 words
   - "findings": 30 to 60 words
   - "technique": 20 to 40 words
   - "consultation_report": 40 to 80 words
7. Use realistic medical style.
8. Do not use placeholders.
9. Any example missing even one mandatory keyword is invalid and must not appear in the final output.

Validation before answering:
- Ensure there are exactly 5 objects.
- Ensure the output is valid JSON.
- Ensure every object contains all mandatory keywords.
- Output only the JSON array.
"""
inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=1200,
        do_sample=True,
        temperature=0.5,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

generated_tokens = output[0][inputs["input_ids"].shape[1]:]
result = tokenizer.decode(generated_tokens, skip_special_tokens=True)
print(result)

/home/test/anaconda3/envs/caise-ner/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


```json
[
    {
        "id": 1,
        "ct_thorax": "A CT thorax scan was performed on a 58-year-old male patient.",
        "findings": "Multiple pulmonary nodules were identified in both lungs, along with a moderate pleural effusion in the right hemithorax and enlarged mediastinal lymph nodes.",
        "technique": "Non-contrast helical CT imaging at 120 kV and 50 mA with 5 mm slice thickness.",
        "consultation_report": "The presence of multiple pulmonary nodules warrants further evaluation via biopsy or follow-up imaging. The pleural effusion suggests potential malignancy or infection, necessitating fluid analysis. Enlarged mediastinal lymph nodes indicate possible metastasis which should be assessed by PET/CT.",
        "keywords_used": ["pulmonary nodule", "pleural effusion", "mediastinal lymph nodes"]
    },
    {
        "id": 2,
        "ct_thorax": "On CT thoracic imaging, several small pulmonary nodules were noted alongside a large pleural effusion in the left lung a

In [3]:
from ncgft.generation.src.load_models import load_local_model
from ncgft.shared.configs.local_run1 import CONFIG

print(load_local_model(CONFIG))



ModuleNotFoundError: No module named 'ncgft'